# Cultural Economics — A Sector You Cannot Size

The IRS Business Master File lists every organisation recognised as tax-exempt — **1,964,958**
of them — and carries `NTEE_CD`, the classification that every sector analysis of the nonprofit
economy runs on. **119,112** organisations sit in NTEE major group A, *Arts, Culture and
Humanities* — **8.56%** of classified organisations — reporting **$56.9 billion** of revenue.

That figure is the one a report would print, and it rests on an assumption the file itself
disproves. **`NTEE_CD` is null for 29.2% of the file — 574,206 organisations — holding $800.1
billion, 18.5% of all the revenue in it.**

Dropping them, which is what computing a sector total over the classified rows does, would be
survivable if they were a random 29.2%. They are not. The unclassified share runs from
**21.2%** among 501(c)(3) charities to **89.7%** in some other subsections, and it is higher in
the smallest revenue quintile than in the largest.

Assume instead that the unclassified are arts at the same rate as the classified — the
missing-at-random assumption, the natural repair — and the sector becomes **$125.4 billion**,
**2.20×** the published-style figure. That assumption is exactly the one the subsection table
rules out.

So this notebook does not produce a number for the nonprofit arts sector. It establishes that
the file cannot produce one, shows the width of what it can bound, and says which assumption
every existing figure is carrying.

## Learning Objectives

1. **Check classification coverage before computing a sector total**, and treat a coding gap as
   a sampling problem rather than a data-cleaning one.
2. **Test whether missingness is random** against a variable you have, rather than assuming it.
3. **Bound a quantity under stated assumptions**, and report the width of the bound as the
   result.
4. **Recognise when the honest output is "this cannot be computed here"**, and say what would be
   needed instead.

## Prerequisites

- D11 (BEA ACPSA) and handbook §1.11 — the other way to size an arts sector, from national
  accounts rather than a registration file. The two measure different things and neither is a
  check on the other.
- Handbook §3 (denominators, universes and partitions).
- Handbook §1.4 (QCEW) for the contrast: there what is missing is flagged per cell. Here what is
  missing is a *classification*, and the row is present and complete in every other respect.

## Data Sources

**Internal Revenue Service, Statistics of Income — Exempt Organizations Business Master File**,
four regional CSVs from `irs.gov/pub/irs-soi`.

**This is a 341 MB download** across four files, with no API and no national single file. The
size is reported in provenance rather than hidden.

Three properties of the file shape everything below:

- **It is a registration file, not a financial one.** `REVENUE_AMT` is present only where a
  return supplied it, is the most recent period on file rather than a common year, and is absent
  for organisations below filing thresholds.
- **NTEE codes are assigned at registration** and are not systematically revised, so a code may
  describe what an organisation was founded to do rather than what it does.
- **It carries no vintage.** No version, no date; the payload hash computed on retrieval is the
  only identifier, as with D17.

## 1. Setup

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    IRS_BMF_REGIONS,
    NTEE_MAJOR_GROUPS,
    fetch_irs_bmf,
)

#: The NTEE major group this notebook is nominally about.
ARTS_GROUP = "A"

#: Subsections to break the missingness down by. 03 is 501(c)(3), the charities
#: almost every nonprofit study restricts itself to.
CHARITY_SUBSECTION = "03"

pd.set_option("display.width", 112)
print(f"regions      : {', '.join(IRS_BMF_REGIONS)}")
print(f"target group : {ARTS_GROUP} -- {NTEE_MAJOR_GROUPS[ARTS_GROUP]}")
print(f"major groups : {len(NTEE_MAJOR_GROUPS)}")


regions      : eo1, eo2, eo3, eo4
target group : A -- Arts, Culture and Humanities
major groups : 26


## 2. The file

In [2]:
"""Retrieve the master file. This is a large download."""
organisations, prov = fetch_irs_bmf()
print(f"organisations : {len(organisations):,}")
print(f"states/areas  : {organisations['STATE'].nunique()}")
print(f"payload hash  : {prov.content_sha256[:16]}...")
print()
for note in prov.notes[1:3]:
    print(f"  - {note}")


organisations : 1,964,958
states/areas  : 62
payload hash  : 8768be5061d3c77a...

  - LARGE BULK DOWNLOAD: 341 MB across 4 files (eo1 49 MB, eo2 126 MB, eo3 165 MB, eo4 1 MB). No API and no national single file.
  - CLASSIFICATION COVERAGE: NTEE_CD is present for 70.8% of rows and absent for 29.2%. Any sector total computed over the classified rows is a total over a subset, and the subset is not random -- the missing share varies strongly by SUBSECTION.


Nearly two million organisations. Note the second provenance line: this is a third
of a gigabyte, and a notebook that pulls it without saying so is misrepresenting its own cost.

## 3. How much of it is classified?

In [3]:
"""How much of the file is classified at all?"""
classified = organisations["is_classified"]
print(f"with an NTEE code : {int(classified.sum()):,} ({classified.mean():.1%})")
print(f"without           : {int((~classified).sum()):,} ({1 - classified.mean():.1%})")
print()
groups = (organisations.loc[classified, "ntee_major"]
          .value_counts()
          .rename_axis("group").reset_index(name="organisations"))
groups["meaning"] = groups["group"].map(NTEE_MAJOR_GROUPS)
print(groups.head(8).to_string(index=False, formatters={"organisations": "{:,.0f}".format}))


with an NTEE code : 1,390,752 (70.8%)
without           : 574,206 (29.2%)

group organisations                                               meaning
    X       205,231                                      Religion-Related
    B       187,152                                             Education
    P       155,311                                        Human Services
    A       119,112                          Arts, Culture and Humanities
    N       114,077                                 Recreation and Sports
    T       105,360 Philanthropy, Voluntarism and Grantmaking Foundations
    S        72,768           Community Improvement and Capacity Building
    O        58,598                                     Youth Development


**70.8% classified, 29.2% not** — 574,206 organisations with no NTEE code at all.

The classified majority breaks down as expected: religion-related, education, human services,
arts and recreation at the top. Arts is the fourth-largest major group by organisation count.

Everything after this point turns on whether those 574,206 can be ignored.

## 4. They cannot

If the unclassified were a random 29.2% of the file, a sector total over the classified rows
would be an unbiased estimate scaled by the classified share. Test it against a variable the file does supply.

In [4]:
"""Is the missing 29% a random 29%?"""
by_subsection = (organisations.groupby("SUBSECTION")
                 .agg(organisations=("is_classified", "size"),
                      unclassified=("is_classified", lambda s: float((~s).mean())))
                 .sort_values("organisations", ascending=False)
                 .head(8))
print(by_subsection.to_string(formatters={
    "organisations": "{:,.0f}".format, "unclassified": "{:.1%}".format}))
print()
charities = organisations["SUBSECTION"] == CHARITY_SUBSECTION
print(f"unclassified among 501(c)({CHARITY_SUBSECTION}) charities : "
      f"{float((~organisations.loc[charities, 'is_classified']).mean()):.1%}")
print(f"unclassified among everything else            : "
      f"{float((~organisations.loc[~charities, 'is_classified']).mean()):.1%}")


           organisations unclassified
SUBSECTION                           
03             1,642,105        21.2%
04                68,449        66.5%
06                57,755        58.3%
07                46,492        62.2%
05                43,643        78.9%
08                32,962        89.7%
19                24,650        73.5%
10                14,276        85.9%



unclassified among 501(c)(03) charities : 21.2%
unclassified among everything else            : 69.8%


**From 21.2% to 89.7%.** Among 501(c)(3) charities — the subsection almost every
nonprofit study restricts itself to — 21.2% lack a code. Among fraternal beneficiary societies
it is 89.7%, among labour and agricultural organisations 78.9%, among social-welfare
organisations 66.5%.

Restricting to charities does not solve it either: **21.2% of them are still unclassified**,
against **69.8%** of everything else. It narrows the problem by roughly a third and leaves one
charity in five outside any sector.

**[rule]** A classification gap is a sampling problem, not a cleaning problem. Before dropping
unclassified rows, test the gap against every variable you have — and if it moves, the rows you
dropped are not the rows you think you dropped.

## 5. And it is related to size

In [5]:
"""And is it related to size?"""
with_revenue = organisations[organisations["REVENUE_AMT"].notna()].copy()
with_revenue["quintile"] = pd.qcut(
    with_revenue["REVENUE_AMT"].rank(method="first"), 5,
    labels=["Q1 smallest", "Q2", "Q3", "Q4", "Q5 largest"],
)
by_size = with_revenue.groupby("quintile", observed=True)["is_classified"].agg(
    organisations="size", classified="mean"
)
print(by_size.to_string(formatters={
    "organisations": "{:,.0f}".format, "classified": "{:.1%}".format}))
print()
print("reporting revenue at all:")
print(organisations.groupby(organisations["REVENUE_AMT"].notna())["is_classified"]
      .agg(organisations="size", classified="mean")
      .to_string(formatters={"organisations": "{:,.0f}".format, "classified": "{:.1%}".format}))


            organisations classified
quintile                            
Q1 smallest       278,244      67.6%
Q2                278,243      79.3%
Q3                278,243      76.8%
Q4                278,243      75.4%
Q5 largest        278,243      73.9%

reporting revenue at all:
            organisations classified
REVENUE_AMT                         
False             573,742      61.5%
True            1,391,216      74.6%


The classification rate is lowest in the smallest revenue quintile — **67.6%** —
against **73.9%** in the largest, and it is not monotone in between, peaking at **79.3%** in the
second quintile.

Organisations reporting no revenue at all are classified **61.5%** of the time against **74.6%**
for those that report some.

The direction matters for what a sector total does. Small and non-reporting organisations are
disproportionately absent from the classified set, so a count of arts organisations from the
classified rows undercounts small ones more than large ones — which is precisely the tail a
cultural-sector study usually cares about.

## 6. The sector, as the file can see it

In [6]:
"""The arts sector, as far as this file can see it."""
arts = organisations[organisations["ntee_major"] == ARTS_GROUP]
unclassified = organisations[~organisations["is_classified"]]
total_revenue = organisations["REVENUE_AMT"].sum()

print(f"organisations in group {ARTS_GROUP} : {len(arts):,} "
      f"({len(arts) / int(classified.sum()):.2%} of classified organisations)")
print(f"their revenue                : ${arts['REVENUE_AMT'].sum() / 1e9:,.1f}B")
print(f"revenue of all classified    : ${organisations.loc[classified, 'REVENUE_AMT'].sum() / 1e9:,.1f}B")
print(f"revenue of the unclassified  : ${unclassified['REVENUE_AMT'].sum() / 1e9:,.1f}B "
      f"({unclassified['REVENUE_AMT'].sum() / total_revenue:.1%} of all revenue in the file)")


organisations in group A : 119,112 (8.56% of classified organisations)


their revenue                : $56.9B
revenue of all classified    : $3,517.2B
revenue of the unclassified  : $800.1B (18.5% of all revenue in the file)


**119,112 organisations, $56.9 billion.** That is the number this file supports if
the unclassified are ignored.

Beside it: the unclassified hold **$800.1 billion**, **18.5%** of all the revenue in the file.
Whatever share of that is arts is missing from the $56.9 billion, and nothing in the file says
what share it is.

## 7. What can be bounded, and how wide it is

In [7]:
"""What can and cannot be said about the sector's size."""
arts_revenue = arts["REVENUE_AMT"].sum()
unclassified_revenue = unclassified["REVENUE_AMT"].sum()
arts_share_of_classified = len(arts) / int(classified.sum())

logical_low = arts_revenue / 1e9
logical_high = (arts_revenue + unclassified_revenue) / 1e9
mar_estimate = (arts_revenue + arts_share_of_classified * unclassified_revenue) / 1e9

print(f"if none of the unclassified are arts : ${logical_low:,.1f}B")
print(f"if all of them are                   : ${logical_high:,.1f}B")
print(f"  the logical bound is {logical_high / logical_low:,.1f}x wide")
print()
print(f"if the unclassified are arts at the same rate as the classified "
      f"({arts_share_of_classified:.2%}):")
print(f"  estimated arts revenue             : ${mar_estimate:,.1f}B")
print(f"  which is {mar_estimate / logical_low:,.2f}x the figure you get by "
      f"ignoring them")


if none of the unclassified are arts : $56.9B
if all of them are                   : $857.0B
  the logical bound is 15.1x wide

if the unclassified are arts at the same rate as the classified (8.56%):
  estimated arts revenue             : $125.4B
  which is 2.20x the figure you get by ignoring them


**Logically, between $56.9B and $857.0B** — a bound **15.1×** wide, and the upper
end is absurd because it assumes every unclassified organisation in America is an arts
organisation. A bound that wide is not a result.

**Under missing-at-random — the natural repair — $125.4B, or 2.20× the naive figure.** That is
the number an analyst would reach for, and section 4 is the reason not to: the missingness is
strongly related to subsection and to size, so MAR is not merely unproven here, it is
contradicted by the file.

So the honest output of this notebook is a refusal. The nonprofit arts sector cannot be sized
from the BMF alone. What would be needed is either a source that codes the whole universe, or an
external estimate of the arts share *within the unclassified*, and the BMF supplies neither.

**[rule]** When a bound is wide and the assumption that narrows it is contradicted by your own
data, publish the refusal rather than the narrowed figure. "Between $56.9B and $857.0B, and the
repair that would narrow it is ruled out here" is a usable statement. "$125.4B" is not.

**And this is not what D11 measured.** BEA's Arts and Cultural Production Satellite Account
sizes arts *production* across the whole economy — commercial and nonprofit, by industry
re-partition — against a definitional boundary that is itself the error bar (handbook §1.11).
This file counts *registered nonprofit organisations* by founding purpose. Two agencies, two
sectors, one word. Neither is a check on the other and neither should be quoted as "the arts
sector" without saying which.

## Limitations

1. **This notebook produces no estimate of the nonprofit arts sector.** That is the finding, not
   an omission.

2. **`REVENUE_AMT` is not a common year.** It is the most recent period on file per
   organisation, so every dollar total here sums across mixed fiscal years and is not a
   time-consistent aggregate.

3. **Revenue is absent for 29.2% of rows for reasons unrelated to NTEE**, including filing
   thresholds. Sections 6 and 7's dollar figures are over organisations that reported, which is
   a second non-random subset stacked on the first.

4. **NTEE major group A is a founding-purpose code**, not a current-activity one, and is not
   systematically revised.

5. **The upper logical bound is deliberately absurd.** It is reported to show the width of what
   the file alone supports, not as a plausible value.

6. **The MAR estimate uses the arts share of classified organisations by count** applied to
   unclassified revenue. Using a revenue-weighted share would give a different figure; neither is
   defensible, which is section 7's point.

7. **No vintage.** The file carries no version or date, so two downloads are comparable only by
   hash.

8. **Territories and foreign addresses are in the `STATE` column** and are not separated here.

## Key Takeaways

1. **29.2% of the IRS master file has no NTEE code** — 574,206 organisations holding $800.1B,
   18.5% of all revenue in the file.

2. **The gap is not random.** 21.2% among 501(c)(3) charities, 89.7% in some other subsections,
   and worse among small and non-reporting organisations.

3. **Restricting to charities does not fix it**: 21.2% of them are still unclassified.

4. **The naive sector figure is $56.9B**; under MAR it is $125.4B, **2.20×** larger. The MAR
   assumption is contradicted by the file.

5. **The logical bound is 15.1× wide**, which is not a result.

6. **Publish the refusal.** A wide bound plus a ruled-out repair is a usable statement; the
   narrowed figure is not.

7. **A classification gap is a sampling problem.** Test it against everything you have before
   dropping a row.

8. **D11 and D22 do not measure the same sector.** Production across the whole economy versus
   registered nonprofits by founding purpose. Neither checks the other.

## Exercises

1. **Find the arts share among the unclassified.** Use organisation names — `NAME` contains
   "MUSEUM", "THEATRE", "ORCHESTRA", "ARTS" — to build a crude classifier, apply it to the
   unclassified rows, and report the arts revenue it finds. Then state every reason that number
   is not the answer.

2. **Test a second non-randomness.** Section 4 uses subsection and section 5 uses size. Test the
   gap against state and against `RULING` date, and report whether either adds anything.

3. **Bound it properly.** Compute Manski-style worst-case bounds on the arts *share* of nonprofit
   revenue, and compare their width to the naive interval in section 7.

4. **Check the founding-purpose problem.** Limitation 4 says NTEE codes are not revised. Using
   `RULING`, compare the arts share among organisations registered before 1990 against those
   registered after 2010, and say what you can conclude.

5. **Reconcile with D11.** State in one paragraph what BEA's ACPSA counts that this file does
   not, and vice versa, with a figure from each.

6. **Write the methods note.** Draft the paragraph a report should carry when it quotes a
   nonprofit sector total from the BMF.

## References

- Internal Revenue Service. *Exempt Organizations Business Master File Extract.*
  https://www.irs.gov/charities-non-profits/exempt-organizations-business-master-file-extract-eo-bmf
- National Center for Charitable Statistics. *NTEE Classification System.*
  https://nccs.urban.org/publication/irs-activity-codes
- Manski, C. F. (1990). "Nonparametric Bounds on Treatment Effects." *American Economic Review*
  80(2): 319–323 — the bounding approach exercise 3 asks for.
- D11 (`08-culture-society/D11_cultural_economics`) and handbook §1.11 — the satellite-account
  measure of the arts sector.
- Handbook §1.4 (suppression with a flag), §3 (denominators, universes and partitions).
- `src/krl_tutorials_data/sources.py` (`fetch_irs_bmf`) and `tests/test_irs_bmf.py`.